In [1]:
import sys
print(sys.executable)

/home/brez/mypy/damod/.venv/bin/python


In [ ]:
#load the dataset
import pandas as pd

df = pd.read_excel("../raw/Book2.xlsx")
df.head()

In [ ]:
#inspect data set
df.info()

1200 rows

17 columns

In [ ]:
df.columns

In [ ]:
#Missing values
df.isnull().sum()

Unnammed 13: 1200 missing

Unnammed 14: 1200 missing

In [ ]:
df.describe()

Confirm duplicated columns

In [ ]:
df = df.drop(columns=[
    "Unnamed: 13",
    "Unnamed: 14",
    "Daily Social Media",
    "StressLevel"
])

df.info()

trim duplicate

In [ ]:
df['gender'].value_counts()

In [ ]:
df['platform_usage'].value_counts()

In [ ]:
df['social_interaction_level'].value_counts()

In [ ]:
df['depression_label'].value_counts()

In [ ]:
df.nunique()

how many uniques each column has

In [ ]:
df['age'].nunique()

'age' has 7 uniques

In [ ]:
df.describe()

summary statistics for numeric columns

In [ ]:
for col in ["gender", "platform_usage", "social_interaction_level", "depression_label"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())

In [ ]:
def usage_category(hours):
    if hours < 3:
        return "Low"
    elif hours < 6:
        return "Moderate"
    else:
        return "High"
    
df["social_media_usage_group"] = df["daily_social_media_hours"].apply(usage_category)

In [ ]:
df["social_media_usage_group"].value_counts()

create categorial group for 'daily_social_media_hours'

In [ ]:
df["mental_strain_score"]= df[[
    "stress_level",
    "anxiety_level",
    "addiction_level"
]].mean(axis=1)

In [ ]:
df[[
    "stress_level",
    "anxiety_level",
    "addiction_level",
    "mental_strain_score"
]].describe()

create 'mental_strain_score'

In [ ]:
df.groupby("social_media_usage_group")[[
    "daily_social_media_hours",
    "sleep_hours",
    "screen_time_before_sleep",
    "academic_performance",
    "physical_activity",
    "stress_level",
    "anxiety_level",
    "addiction_level",
    "mental_strain_score"
]].mean().round(2)

higher daily social media hours clearly does not mean worse mental strain

In [ ]:
df.groupby("depression_label")[[
    "daily_social_media_hours",
    "sleep_hours",
    "screen_time_before_sleep",
    "academic_performance",
    "stress_level",
    "anxiety_level",
    "addiction_level",
]].mean()

Compare average behavior and wellbeing indicators between depression label groups. 
The depression-indicator group shows higher average social media usage, lower sleep hours, and higher stress/anxiety, but the group is small, so results is biased towards total population

In [ ]:
df.groupby("platform_usage")[[
    "daily_social_media_hours",
    "sleep_hours",
    "screen_time_before_sleep",
    "academic_performance",
    "physical_activity",
    "stress_level",
    "anxiety_level",
    "addiction_level",
    "mental_strain_score"
]].mean().round(2)

Separating platform doesn't mean anything either, look at the mental_strain_score

In [ ]:
df.groupby("social_interaction_level")[[
    "daily_social_media_hours",
    "sleep_hours",
    "screen_time_before_sleep",
    "academic_performance",
    "physical_activity",
    "stress_level",
    "anxiety_level",
    "addiction_level",
    "mental_strain_score"
]].mean().round(2)

Same thing

In [ ]:
numeric_cols = [
    "age",
    "daily_social_media_hours",
    "sleep_hours",
    "screen_time_before_sleep",
    "academic_performance",
    "physical_activity",
    "stress_level",
    "anxiety_level",
    "addiction_level",
    "mental_strain_score"
]

df[numeric_cols].corr().round(2)

Almost all correlations between behavior and lifestyle are near zero. No meaningful linear relationship

So far we have learned:

1. Different social media usage group have almost identical mental strain scores, which means daily social media duration is not a strong separator

2. Platform usage have similar mental_strain_score

3. Social interaction level groups show similar mental_strain_score

4. Correlation matrix show near no relationship

All this tells us that obvious variables are not strongly explanatory

Dashboard story:
Exploratory Dashboard of Teen Social Media Behavior and Wellbeing Indicators

This dashboard examines whether social media usage, platform choice, sleep, academic performance, and lifestyle indicators show visible patterns with mental strain. The analysis finds that most broad behavior groups are relatively similar, while the depression-indicator subgroup shows sharper difference in sleep, stress, anxiety, and usage

In [ ]:
overview_metrics = {
    "total_records": len(df),
    "avg_social_media_hours": float(round(df["daily_social_media_hours"].mean(), 2)),
    "avg_sleep_hours": float(round(df["sleep_hours"].mean(), 2)),
    "avg_mental_strain_score": float(round(df["mental_strain_score"].mean(), 2)),
    "depression_indicator_rate": float(round(df["depression_label"].mean() * 100, 2))
}

overview_metrics

Usage summary:

In [ ]:
usage_group_summary = df.groupby("social_media_usage_group").agg(
    records=("social_media_usage_group", "count"),
    avg_social_media_hours=("daily_social_media_hours", "mean"),
    avg_sleep_hours=("sleep_hours", "mean"),
    avg_academic_performance=("academic_performance", "mean"),
    avg_mental_strain_score=("mental_strain_score", "mean")
).round(2)

usage_group_summary

Daily social media duration alone does not clearly separate mental strain, sleep, or academic performance in this dataset

Platform summary:

In [ ]:
platform_summary = df.groupby("platform_usage").agg(
    records=("platform_usage", "count"),
    avg_social_media_hours=("daily_social_media_hours", "mean"),
    avg_sleep_hours=("sleep_hours", "mean"),
    avg_academic_performance=("academic_performance", "mean"),
    avg_mental_strain_score=("mental_strain_score", "mean")
).round(2)

platform_summary

Platform usage also does not show a strong difference in mental strain or academic performance

mental strain score summary grouped by depression label:

In [ ]:
depression_summary = df.groupby("depression_label").agg(
    records=("depression_label", "count"),
    avg_social_media_hours=("daily_social_media_hours", "mean"),
    avg_sleep_hours=("sleep_hours", "mean"),
    avg_stress_level=("stress_level", "mean"),
    avg_anxiety_level=("anxiety_level", "mean"),
    avg_addiction_level=("addiction_level", "mean"),
    avg_mental_strain_score=("mental_strain_score", "mean")
).round(2)

depression_summary

Broad social media behavior variables such as usage duration and platform choice show limited separation across wellbeing indicators. But the small depression-indicator subgroup shows noticeably higher strain and lower sleep, making it an interesting story for the dashboard